In [ ]:
import os
import re
import json
import pandas as pd
from typing import TypedDict, List, Dict, Any

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from crewai_tools import ExaSearchTool, ScrapeWebsiteTool

os.environ["EXA_API_KEY"] = "f3b38ea4-13ab-4340-ba45-abcf5f692087"
os.environ["COMPOSIO_API_KEY"] = "ak_Y6g-gZuaY7CJa1-sIsNV"  

from composio import Composio
composio = Composio(api_key="ak_Y6g-gZuaY7CJa1-sIsNV")

""" Only used for linkedin setup once """
# Create a session for the user
#session = composio.create(user_id="simardhingra2660_workspace")

# Initiate authorization for LinkedIn
#connection_request = session.authorize("linkedin")

#print(f"Please visit this URL to log in and authorize LinkedIn:\n{connection_request.redirect_url}")


In [ ]:


# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================

# Initialize Local LLM & Embeddings
ollama_llm = ChatOllama(model="qwen2.5:7b", base_url="http://localhost:11434")
ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")

# External Tools
exa_search = ExaSearchTool()
scrape_tool = ScrapeWebsiteTool()

# Resume Setup
PDF_PATH = r"C:\Users\simar\Downloads\Simar's_CV_PM_B2C.pdf"
CHROMA_PERSIST_DIR = r"C:\Users\simar\Downloads\chroma_db_resumes"
EXCEL_PATH = r"C:\Users\simar\Downloads\Matched_Jobs.xlsx"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load_and_split()
vectorstore = Chroma.from_documents(docs, ollama_embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})



# 3. Load from Disk if Exists, Else Embed Once
if os.path.exists(CHROMA_PERSIST_DIR) and os.listdir(CHROMA_PERSIST_DIR):
    print("⚡ Loading existing ChromaDB embeddings from disk (Instant)...")
    vectorstore = Chroma(
        persist_directory=CHROMA_PERSIST_DIR,
        embedding_function=ollama_embeddings
    )
else:
    print("📦 First run: Converting PDF to embeddings & saving to ChromaDB...")
    loader = PyPDFLoader(PDF_PATH)
    docs = loader.load_and_split()
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=ollama_embeddings,
        persist_directory=CHROMA_PERSIST_DIR
    )
    print("✓ Embeddings saved to disk successfully!")

# Retrieve only the 2 most relevant chunks to keep the LLM context prompt concise
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# ==========================================
# 2. DEFINE GRAPH STATE
# ==========================================

class PipelineState(TypedDict):
    candidate_name: str
    role_query: str
    target_geos: str
    experience_level: str
    job_duration: str
    resume_summary: str
    raw_jobs: List[Dict[str, Any]]
    evaluated_jobs: List[Dict[str, Any]]
    contacts: List[Dict[str, Any]]


# ==========================================
# 3. DEFINE GRAPH NODES
# ==========================================

# Node 1: Parse Resume PDF using Local RAG
def parse_resume_node(state: PipelineState) -> Dict[str, Any]:
    print("📄 Node 1: Retrieving Resume Profile from ChromaDB...")
    
    # Query ChromaDB (Takes ~10-20ms)
    relevant_docs = retriever.invoke(f"Extract key skills, job titles, domain, company names and location for {state['candidate_name']}")
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    # Concise prompt to reduce token generation time on CPU
    prompt = f"""Summarize this candidate resume context into concise bullet points covering:
1. Technical Skills
2. Job Titles Held
3. Domain Experience 
4. Description of type of Work done in less than 100 words: 0-1 Product building, BUsiness Product, Technical Product management etc.

Resume Context:
{context}"""
    
    response = ollama_llm.invoke(prompt)
    return {"resume_summary": response.content}


# Node 2: Search Jobs using Exa Search

def search_jobs_node(state: PipelineState) -> Dict[str, Any]:
    user_duration = state.get("job_duration", "1 week")
    print(f"🔍 Node 2: Searching LinkedIn Jobs via Composio (Posted within last {user_duration})...")
    
    # 1. Execute Composio LinkedIn Search Action
    exa_query = f"{state['role_query']} in {state['target_geos']} posted in last {user_duration}"
    raw_web_jobs = exa_search.run(exa_query)
    
    linkedin_jobs = ""
    try:
        linkedin_res = composio.execute_action(
            action="LINKEDIN_SEARCH_JOBS",  # or "LINKEDIN_SEARCH"
            params={"query": f"{role_query} {target_geos}"},
            user_id="simardhingra2660_workspace"
        )
        linkedin_jobs = str(linkedin_res)
    except Exception as e:
        search_results = f"Composio LinkedIn search fallback. Error: {str(e)}"

    # 2. Extract structured listings using Ollama
    prompt = f"""Extract and structure a JSON list of job postings from these search outputs:

    EXA RESULTS:
    {raw_web_jobs}

    LINKEDIN RESULTS:
    {linkedin_jobs}

    Format each object with keys:
    - "Job Profile"
    - "Job Link"
    - "Platform Name"
    - "Date Posted"
"""

    resp = ollama_llm.invoke(prompt)
    try:
        json_match = re.search(r'\[.*\]', response.content, re.DOTALL)
        parsed_jobs = json.loads(json_match.group(0)) if json_match else []
    except Exception:
        parsed_jobs = []
        
    return {"raw_jobs": parsed_jobs}
    """
    try:
        json_match = re.search(r'\[.*\]', resp.content, re.DOTALL)
        jobs = json.loads(json_match.group(0)) if json_match else []
    except Exception:
        jobs = []

    formatted_jobs = []
    for item in jobs:
        formatted_jobs.append({
            "Job Profile": item.get("Job Profile", item.get("title", "N/A")),
            "Job Link": item.get("Job Link", item.get("link", "N/A")),
            "Platform Name": "LinkedIn",
            "Date Posted": item.get("Date Posted", f"Within last {user_duration}")
        })

    return {"raw_jobs": formatted_jobs}
"""
    
# Node 3: Evaluate Match Scores
def evaluate_matches_node(state: PipelineState) -> Dict[str, Any]:
    print("⚖️ Node 3: Evaluating All Job Matches in a Single Batch")
    raw_jobs = state.get("raw_jobs", [])
    
    if not raw_jobs:
        return {"evaluated_jobs": []}

    # Format all jobs into a single clean text block
    jobs_text = "\n".join([
        f"ID {i+1}: {job.get('Job Profile')} | Link: {job.get('Job Link')} | Posted: {job.get('Date Posted')}"
        for i, job in enumerate(raw_jobs)
    ])

    prompt = f"""Compare the candidate profile from resume context against ALL the job postings listed below.

Candidate Profile:
{state['resume_summary']}

Job Postings to Evaluate:
{jobs_text}

For EACH job ID, assign a match rating (Poor, Average, Good, Perfect) and a brief rationale under 20 words.
Return strictly a valid JSON array matching this format:
[
  {{"id": 1, "rating": "Good", "rationale": "Strong alignment with B2C Fintech experience."}},
  {{"id": 2, "rating": "Poor", "rationale": "Requires hardware engineering background."}}
]"""

    resp = ollama_llm.invoke(prompt)
    
    # Parse batch JSON output
    try:
        json_match = re.search(r'\[.*\]', resp.content, re.DOTALL)
        evaluations = json.loads(json_match.group(0)) if json_match else []
        eval_dict = {item.get("id"): item for item in evaluations}
    except Exception:
        eval_dict = {}

    # Merge ratings back into job dicts
    evaluated_jobs = []
    for i, job in enumerate(raw_jobs):
        eval_data = eval_dict.get(i + 1, {})
        job["rating"] = eval_data.get("rating", "Average")
        job["rationale"] = eval_data.get("rationale", "Evaluated based on profile keyword alignment.")
        evaluated_jobs.append(job)

    return {"evaluated_jobs": evaluated_jobs}


    
# Node 4: Pure Python Excel Exporter (FAST: Zero LLM overhead)

def export_excel_node(state: PipelineState) -> Dict[str, Any]:
    print("📊 Node 4: Writing Jobs to Excel (Python)...")
    valid_jobs = [j for j in state.get("evaluated_jobs", []) if j.get("rating") in ["Average", "Good", "Perfect"]]
    
    new_rows = []
    for j in valid_jobs:
        new_rows.append({
            "Job Profile": j.get("Job Profile", "N/A"),
            "Job Link": j.get("Job Link", "N/A"),
            "Platform Name": j.get("Platform Name", "N/A"),
            "Date Posted": j.get("Date Posted", "N/A")
        })
    
    new_df = pd.DataFrame(new_rows, columns=["Job Profile", "Job Link", "Platform Name", "Date Posted"])
    
    # Read existing file if present and concatenate
    if os.path.exists(EXCEL_PATH):
        try:
            existing_df = pd.read_excel(EXCEL_PATH, sheet_name="Matched Jobs")
            # Combine old and new jobs, removing duplicates if needed
            combined_df = pd.concat([existing_df, new_df], ignore_index=True).drop_duplicates(subset=["Job Link"])
        except Exception:
            combined_df = new_df
    else:
        combined_df = new_df

    combined_df.to_excel(EXCEL_PATH, index=False, sheet_name="Matched Jobs")
    print(f"  ✓ Excel file updated with {len(combined_df)} total jobs at: {EXCEL_PATH}")
    return {}


# Node 5: Find LinkedIn Contacts
def find_contacts_node(state: PipelineState) -> Dict[str, Any]:
    print("🤝 Node 5: Searching Senior Hiring Contacts via Composio LinkedIn...")
    contacts = []
    top_jobs = [j for j in state.get("evaluated_jobs", []) if j.get("rating") in ["Good", "Perfect", "Average"]]
    
    for job in top_jobs[:10]:
        company = job.get("Target Company")
        target_role = job.get("Job Profile", "Product Manager") # Get role from job or fallback
        search_query = f"3 profiles of either Hiring Manager OR Talent Acquisition OR VP Product Management OR VP Tech at {company}"
        
        try:
            # Execute Composio LinkedIn Profile/People Search
            response = composio.execute_action(
                action=Action.LINKEDIN_SEARCH_PEOPLE_BY_PROFILE_OR_COMPANY,
                params={"query": search_query},
                user_id="simardhingra2660_workspace"
            )
            contacts.append({
                "company": company,
                "target_role": target_role,
                "linkedin_results": response})
       
        except Exception as e:
            print(f"⚠️ LinkedIn contact search failed for {company}: {e}")
            contacts.append({
                "company": company,
                "target_role": role,
                "linkedin_results": "Not found"          
         })
        
    return {"contacts": contacts}


# Node 6: Append Outreach Columns to Excel (Pure Python)
def append_outreach_node(state: PipelineState) -> Dict[str, Any]:
    print("📧 Node 6: Appending Outreach Data to Excel (Python)...")
    if os.path.exists(EXCEL_PATH):
        df = pd.read_excel(EXCEL_PATH, sheet_name="Matched Jobs")
    else:
        df = pd.DataFrame()

    outreach_summary = "\n".join([f"{c['Company']}: Search details generated" for c in state.get("contacts", [])])
    
    df["Prospect_Employer_Profile_mail"] = outreach_summary
    df["Prospect_Employer_Profile_LI"] = outreach_summary
    df.to_excel(EXCEL_PATH, index=False, sheet_name="Matched Jobs")
    
    print(f"  ✓ Excel updated with outreach columns at: {EXCEL_PATH}")
    return {}







In [ ]:
# ==========================================
# 4. BUILD THE GRAPH PIPELINE
# ==========================================

workflow = StateGraph(PipelineState)

# Add Nodes
workflow.add_node("parse_resume", parse_resume_node)
workflow.add_node("search_jobs", search_jobs_node)
workflow.add_node("evaluate_matches", evaluate_matches_node)
workflow.add_node("export_excel", export_excel_node)
workflow.add_node("find_contacts", find_contacts_node)
workflow.add_node("append_outreach", append_outreach_node)

# Construct Sequential & Parallel Graph Edges
workflow.add_edge(START, "parse_resume")
workflow.add_edge("parse_resume", "search_jobs")
workflow.add_edge("search_jobs", "evaluate_matches")

# Fan-out: Run Excel Export & Contact Finding in Parallel
workflow.add_edge("evaluate_matches", "export_excel")
workflow.add_edge("evaluate_matches", "find_contacts")

# Fan-in: Combine outputs into final Excel update
workflow.add_edge("export_excel", "append_outreach")
workflow.add_edge("find_contacts", "append_outreach")
workflow.add_edge("append_outreach", END)

app = workflow.compile()

In [ ]:
# ==========================================
# 5. EXECUTE THE PIPELINE
# ==========================================

print("🚀 Starting LangGraph Pipeline Execution...\n")
job_duration_input = input("Enter posting recency/duration (e.g., '1 week', '24 hours', '1 month'): ").strip() or "1 week"

inputs = {
    "candidate_name": "Simar",
    "role_query": "AI/ML Product Manager & Product Builder & AI Builder & Chief of Staff",
    "target_geos": "London, Dubai, Singapore, or Remote",
    "experience_level": "Senior (5+ years)",
    "job_duration": job_duration_input  # <--- Passed to pipeline
}


final_state = app.invoke(inputs)

print("\n✓ LangGraph Execution Complete!")
print("\nSummary of Extracted Resume Profile:\n", final_state.get("resume_summary"))

### job application agent

agent 1- a RAG agent which has access to my profile pdf, is able to extract relevant information from it like job titles, geo, years of experience, technical capability
Role- Senior Knowdlegdge management Specialist  
Goal- Converting the knowledge into legible, readible and comprehendible structured bullet points without changing any meaning or coherency.  
Backstory- You are a meticulous researcher. You specialize in parsing local documentation
        to answer queries without introducing outside hallucinations.

Task- to process the GIVEN PDF docs without editing anything and storing in memory

(I want to use multiple documents - here resumes)




agent 2 - a job researcher which is able to search for relevant jobs for {candidate_name} specific geo/profile/experience 
domain/industry etc.
Role- A specialist career job finder
Goal- to look for relevant jobs for the profiles provided  as per the user query: {user_query}
Backstory- You have successfully managed to search jobs from all known platforms which may have a job posting  (like LinkedIn/IIm jobs/wellfound, other social media platform posts, public forum posts by individuals and all local job posting sites for each geography provided)

Task- based on user query, find jobs posted online across different job listing platforms across organisations of all sizes posted at any duration
Output- A list of 25 jobs with their job description links 


agent3- a verifier which provides a similarity score b/w my profile doc and job description and background 
and verifies if the score is above a certain threshold
Role- Job Profile Evaluator in a Talent Acquisition Team
Goal- To search for relevancy of profile match 
Backstory- You have a background of matching candidate profiles for organisations ranging from new age startups to big tech and fortune 500 companies across globe. You also check if the job posting is updated, posted by a legitimate company/org

Task- Using the the links given, evaluate which job description title matches the agent 1 document. Cite a brief reason for match in less than 50 words
Output- a score out of Poor/Average/Good/Perfect match for profiles matching my resume. Cite a reason for match in less than 50 words 



agent 4- creates a list of jobs posted in the previous x duration 
Format should be excel with -
Role- Executive Assitant & Data Entry specialist
Goal- You enter data fetched to a legible/readable and structured format like excel for humans to process 
Backstory- You have history of reading and scraping useful information into a human readable structure & format. You are an expert at Excel 

Task- Convert the job matches with Average or higher score in an excel table to be accessed from system
Output-An excel table with the following columns:
1.JOb Profile- scrape the Title of the job
2.Job Link- the web link to directly access the job description
3. Platform Name- fetch/scrape the name of the platform where it is posted
4. Date posted- fetch the date of job post
Append all matched jobs to the above mentioned columns. 

Tools- excel


agent 5 - takes the input as csv and starts finding people on linkedIn who are senior to me or might be hiring in that org, anyone related to tech or management roles near to my role
Role- Networking Specialist
Goal- Headhunting best profiles via using web search
Backstory- You have a track record of hunting down profiles from all social media platforms who have or are likely to hire for relevant roles

Task- Based on the best profile matches, find 3 relevant profiles who are in either management roles or are senior in roles {candidate_name} is looking for
Output- A plan for cold reachout for the mentioned profiles



agent 6 -returns the best profile linkedin URLs + retrieves their email id using appolo as a tool, adds it in another column of the same excel
Role- data entry Specialist  
Goal- You enter data fetched to a legible/readable and structured format like excel for humans to process 
 
Backstory- You have a background in sales navigation and finding ways via official emails/ linkedIn profile links to reach out to prospect employers

Task-To retrieve emails and LinkedIn profiles for people selected by the agent 5
Output- add a column in the given excel with name "Prospect_Employer_Profile_mail" & "Prospect_Employer_Profile_LI" 
A list of 3 LinkedIn profile URLs & their official emails- each in a single cell 
Tools- LinkedIn_Search or google search or apollo.io search (whichever is accessible)




in London, Spain, Netherlands, Singapore, India or any Remote Roles
 